In [22]:

# ─────────────────────────────────────────────────────────────
# CELL 1 — Install dependencies
# ─────────────────────────────────────────────────────────────
# Run this cell first. It may take 2–3 minutes.

!pip -q install --upgrade pip
!pip -q install opencv-python-headless pillow tqdm
!pip -q install transformers accelerate safetensors timm einops
!pip -q install supervision==0.22.0
!pip -q install scikit-learn

print("✅ Dependencies installed")



✅ Dependencies installed


In [23]:
from google.colab import files
uploaded = files.upload()                      # pick your .mp4 file
VIDEO_PATH = list(uploaded.keys())[0]

Saving 2BBwMYE-FC4_00.mp4 to 2BBwMYE-FC4_00.mp4


In [24]:
print(f"✅ Video set to: {VIDEO_PATH}")

✅ Video set to: 2BBwMYE-FC4_00.mp4


In [25]:

# ─────────────────────────────────────────────────────────────
# CELL 3 — Imports & device setup
# ─────────────────────────────────────────────────────────────
import os, sys, math, warnings, gc
from itertools import combinations
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import supervision as sv
from transformers import (
    AutoProcessor,
    AutoModelForZeroShotObjectDetection,
    VideoMAEModel,
    VideoMAEImageProcessor,
)

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Running on: {DEVICE}")
if DEVICE == "cpu":
    print("⚠️  No GPU detected — inference will be slow (~10–30 min). "
          "Go to Runtime → Change runtime type → GPU for speed.")



✅ Running on: cuda


In [26]:

# ─────────────────────────────────────────────────────────────
# CELL 4 — Load models
# ─────────────────────────────────────────────────────────────
print("Loading GroundingDINO …")
GROUNDING_ID  = "IDEA-Research/grounding-dino-base"
TEXT_PROMPT   = "car. bus. truck. motorcycle. bicycle. rickshaw. van. auto rickshaw."
BOX_THRESHOLD = 0.28
TEXT_THRESHOLD = 0.25
RESIZE_LONG_EDGE = 640

processor_gd = AutoProcessor.from_pretrained(GROUNDING_ID)
detector_gd  = AutoModelForZeroShotObjectDetection.from_pretrained(GROUNDING_ID).to(DEVICE).eval()

print("Loading VideoMAE-v2 …")
VMAE_ID         = "MCG-NJU/videomae-base"
VMAE_NUM_FRAMES = 16
VMAE_DIM        = 768
vmae_proc  = VideoMAEImageProcessor.from_pretrained(VMAE_ID)
vmae_model = VideoMAEModel.from_pretrained(VMAE_ID).to(DEVICE).eval()

print("✅ Models loaded")


# ─────────────────────────────────────────────────────────────
# CELL 5 — Core pipeline functions
# ─────────────────────────────────────────────────────────────

# ── Detection ────────────────────────────────────────────────
def resize_keep_aspect(img, long_edge):
    h, w = img.shape[:2]
    scale = long_edge / max(h, w)
    if scale >= 1.0:
        return img, 1.0
    nh, nw = int(round(h * scale)), int(round(w * scale))
    return cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR), scale

@torch.no_grad()
def detect_objects(frame_rgb):
    fr, scale = resize_keep_aspect(frame_rgb, RESIZE_LONG_EDGE)
    inputs = processor_gd(images=fr, text=TEXT_PROMPT, return_tensors="pt").to(DEVICE)
    outputs = detector_gd(**inputs)
    target_sizes = torch.tensor([fr.shape[:2]], device=DEVICE)
    try:
        res = processor_gd.post_process_grounded_object_detection(
            outputs, inputs.input_ids, BOX_THRESHOLD, TEXT_THRESHOLD, target_sizes)[0]
    except TypeError:
        try:
            res = processor_gd.post_process_grounded_object_detection(
                outputs, inputs.input_ids, target_sizes, BOX_THRESHOLD, TEXT_THRESHOLD)[0]
        except TypeError:
            res = processor_gd.post_process_grounded_object_detection(
                outputs, inputs.input_ids, target_sizes)[0]
    if len(res.get("boxes", [])) == 0:
        return sv.Detections.empty()
    boxes  = res["boxes"].detach().cpu().numpy()
    scores = res["scores"].detach().cpu().numpy()
    keep   = scores >= BOX_THRESHOLD
    boxes, scores = boxes[keep], scores[keep]
    if len(boxes) == 0:
        return sv.Detections.empty()
    if scale != 1.0:
        boxes = boxes / scale
    return sv.Detections(
        xyxy=boxes.astype(np.float32),
        confidence=scores.astype(np.float32),
        class_id=np.zeros(len(boxes), dtype=np.int32),
    )

# ── VideoMAE feature extraction ──────────────────────────────
@torch.no_grad()
def extract_vmae_features(clip_frames_rgb):
    if len(clip_frames_rgb) < VMAE_NUM_FRAMES:
        while len(clip_frames_rgb) < VMAE_NUM_FRAMES:
            clip_frames_rgb = clip_frames_rgb + [clip_frames_rgb[-1]]
    elif len(clip_frames_rgb) > VMAE_NUM_FRAMES:
        idxs = np.linspace(0, len(clip_frames_rgb)-1, VMAE_NUM_FRAMES).astype(int)
        clip_frames_rgb = [clip_frames_rgb[i] for i in idxs]
    inputs = vmae_proc(clip_frames_rgb, return_tensors="pt").to(DEVICE)
    out    = vmae_model(**inputs)
    return out.last_hidden_state.mean(dim=1).squeeze(0).cpu().numpy()

# ── Geometry helpers ─────────────────────────────────────────
def center(bb):
    return np.array([(bb[0]+bb[2])/2, (bb[1]+bb[3])/2], dtype=np.float32)

def iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0., ix2-ix1), max(0., iy2-iy1)
    inter = iw * ih
    aa = max(0., a[2]-a[0]) * max(0., a[3]-a[1])
    ab = max(0., b[2]-b[0]) * max(0., b[3]-b[1])
    return float(inter / (aa + ab - inter + 1e-9))

def impact_point_from_boxes(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    if ix2 > ix1 and iy2 > iy1:
        return np.array([(ix1+ix2)/2, (iy1+iy2)/2], dtype=np.float32)
    ca, cb = center(a), center(b)
    pa = np.array([np.clip(cb[0], a[0], a[2]), np.clip(cb[1], a[1], a[3])], dtype=np.float32)
    pb = np.array([np.clip(ca[0], b[0], b[2]), np.clip(ca[1], b[1], b[3])], dtype=np.float32)
    return (pa + pb) / 2.0

# ── Tracking ─────────────────────────────────────────────────
DETECT_FPS    = 2.0
MAX_VIDEO_SEC = 30.0   # increase if your video is longer

def load_frames(path, max_seconds=None):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames, max_f = [], int((max_seconds * fps) if max_seconds else 1e18)
    while True:
        ok, bgr = cap.read()
        if not ok or len(frames) >= max_f:
            break
        frames.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames, float(fps)

def run_tracking(frames, fps, detect_fps=DETECT_FPS):
    stride   = max(1, int(round(fps / detect_fps)))
    tracker  = sv.ByteTrack(
        track_activation_threshold=0.25,
        lost_track_buffer=30,
        minimum_matching_threshold=0.8,
        frame_rate=int(round(fps)),
    )
    det = {'frames': [], 'bboxes': [], 'track_ids': []}
    trajectories = defaultdict(lambda: {'frames': [], 'bboxes': [], 'velocities': []})

    for idx in tqdm(range(0, len(frames), stride), desc="Detecting & Tracking"):
        d    = detect_objects(frames[idx])
        d_tr = tracker.update_with_detections(d)
        if d_tr.tracker_id is None or len(d_tr) == 0:
            det['frames'].append(idx)
            det['bboxes'].append(np.empty((0, 4), dtype=np.float32))
            det['track_ids'].append([])
            continue
        bbs  = d_tr.xyxy.astype(np.float32)
        tids = [int(t) for t in d_tr.tracker_id]
        det['frames'].append(idx)
        det['bboxes'].append(bbs)
        det['track_ids'].append(tids)
        for bb, tid in zip(bbs, tids):
            c    = center(bb)
            traj = trajectories[tid]
            vel  = ([0.0, 0.0] if not traj['frames'] else
                    ((c - center(traj['bboxes'][-1])) /
                     max(1e-3, (idx - traj['frames'][-1]) / fps)).tolist())
            traj['frames'].append(idx)
            traj['bboxes'].append(bb.copy())
            traj['velocities'].append(vel)
    return det, dict(trajectories)

# ── Interaction scoring ───────────────────────────────────────
W_IOU, W_CLOSING, W_DECEL = 1.5, 1.0, 0.6

def compute_interaction_scores(det, trajectories, fps):
    scores = []
    for k, f in enumerate(det['frames']):
        bbs, tids = det['bboxes'][k], det['track_ids'][k]
        if len(tids) < 2:
            scores.append((f, 0.0, None))
            continue
        best_s, best_pair = 0.0, None
        for i, j in combinations(range(len(tids)), 2):
            if tids[i] == tids[j]:
                continue
            s_iou = iou_xyxy(bbs[i], bbs[j])
            s_close = 0.0
            if k > 0:
                pt = det['track_ids'][k-1]
                pb = det['bboxes'][k-1]
                if tids[i] in pt and tids[j] in pt:
                    pi, pj = pt.index(tids[i]), pt.index(tids[j])
                    d_prev = np.linalg.norm(center(pb[pi]) - center(pb[pj]))
                    d_now  = np.linalg.norm(center(bbs[i]) - center(bbs[j]))
                    s_close = max(0., d_prev - d_now)
            def _decel(tid):
                tr = trajectories.get(tid)
                if tr is None or len(tr['velocities']) < 2:
                    return 0.0
                return max(0., np.linalg.norm(tr['velocities'][-2]) -
                               np.linalg.norm(tr['velocities'][-1]))
            s = W_IOU * s_iou + W_CLOSING * s_close + W_DECEL * (_decel(tids[i]) + _decel(tids[j]))
            if s > best_s:
                best_s, best_pair = s, (tids[i], tids[j])
        scores.append((f, best_s, best_pair))
    return scores

# ── Feature sequence ─────────────────────────────────────────
CANDIDATE_WINDOW_SEC = 3.0

def build_feature_sequence(frames, fps, det, trajectories,
                            clip_stride_sec=1.0, candidate_frames=None):
    clip_stride = max(1, int(clip_stride_sec * fps))
    window = max(1, int(VMAE_NUM_FRAMES * (fps / 8.0)))
    H, W   = frames[0].shape[:2]
    T      = len(frames)
    frame_to_di = {f: i for i, f in enumerate(det['frames'])}

    if candidate_frames:
        starts, radius = set(), int(CANDIDATE_WINDOW_SEC * fps)
        for cf in candidate_frames:
            for s in range(max(0, cf - radius), min(T - window, cf + radius) + 1, clip_stride):
                starts.add(s)
        starts = sorted(starts)
    else:
        starts = list(range(0, T - window + 1, clip_stride))

    feats, stamps = [], []
    for start in tqdm(starts, desc="Extracting VideoMAE features", leave=False):
        end = start + window
        if end > T:
            continue
        mid  = (start + end) // 2
        idxs = np.linspace(start, end - 1, VMAE_NUM_FRAMES).astype(int)
        clip = [frames[i] for i in idxs]

        roi = None
        for det_f in det['frames']:
            if abs(det_f - mid) <= clip_stride:
                di = frame_to_di.get(det_f)
                if di is not None and len(det['bboxes'][di]) >= 2:
                    all_bb = det['bboxes'][di]
                    x1 = max(0, int(all_bb[:, 0].min()) - 20)
                    y1 = max(0, int(all_bb[:, 1].min()) - 20)
                    x2 = min(W, int(all_bb[:, 2].max()) + 20)
                    y2 = min(H, int(all_bb[:, 3].max()) + 20)
                    if (x2 - x1) > 40 and (y2 - y1) > 40:
                        roi = (x1, y1, x2, y2)
                break
        clip = ([cv2.resize(f[roi[1]:roi[3], roi[0]:roi[2]], (224, 224)) for f in clip]
                if roi else [cv2.resize(f, (224, 224)) for f in clip])

        feats.append(extract_vmae_features(clip))
        stamps.append(mid / fps)

    if not feats:
        idxs = np.linspace(0, T - 1, VMAE_NUM_FRAMES).astype(int)
        feats.append(extract_vmae_features([cv2.resize(frames[i], (224, 224)) for i in idxs]))
        stamps.append(T / (2 * fps))

    return np.stack(feats, axis=0), np.array(stamps, dtype=np.float32)

# ── TriDet head ───────────────────────────────────────────────
class TriDetHead(nn.Module):
    def __init__(self, feat_dim=VMAE_DIM, hidden=256, n_layers=3):
        super().__init__()
        layers, in_d = [], feat_dim
        for _ in range(n_layers):
            layers.extend([nn.Conv1d(in_d, hidden, 3, padding=1), nn.ReLU(inplace=True)])
            in_d = hidden
        self.backbone = nn.Sequential(*layers)
        self.head_d1 = nn.Conv1d(hidden, 3, 3, padding=1,  dilation=1)
        self.head_d2 = nn.Conv1d(hidden, 3, 3, padding=2,  dilation=2)
        self.head_d4 = nn.Conv1d(hidden, 3, 3, padding=4,  dilation=4)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        h = self.backbone(x)
        out = ((self.head_d1(h) + self.head_d2(h) + self.head_d4(h)) / 3.0).permute(0, 2, 1)
        return out[:, :, 0].sigmoid(), out[:, :, 1:].relu()

tridet = TriDetHead().to(DEVICE).eval()

@torch.no_grad()
def tridet_predict_time(feats, stamps):
    x       = torch.tensor(feats, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    conf, dt = tridet(x)
    conf    = conf.squeeze(0).cpu().numpy()
    dt      = dt.squeeze(0).cpu().numpy()
    if conf.max() < 0.1:
        return float(stamps[len(stamps) // 2])
    top_idx = np.argsort(conf)[-min(5, len(conf)):]
    times   = [(stamps[i] - dt[i, 0] + stamps[i] + dt[i, 1]) / 2 for i in top_idx]
    wts     = conf[top_idx]
    wts    /= wts.sum() + 1e-9
    return float(np.dot(times, wts))

def fallback_peak_time(scores, fps):
    if not scores:
        return 0.0
    vals = np.array([s[1] for s in scores], dtype=np.float32)
    k    = min(7, len(vals))
    sm   = np.convolve(vals, np.ones(k) / k, mode="same") if k >= 3 else vals
    return float(scores[int(np.argmax(sm))][0] / fps)

# ── Collision pair finding ────────────────────────────────────
SPEED_THRESHOLD = 3.0

def find_colliding_pair(det, trajectories, acc_frame, fps):
    frame_to_idx  = {f: i for i, f in enumerate(det['frames'])}
    window_frames = [f for f in det['frames'] if abs(f - acc_frame) <= int(fps * 1.0)]
    best_iou, best_pair = 0.0, None
    for f in window_frames:
        fi = frame_to_idx[f]
        bboxes, tids = det['bboxes'][fi], det['track_ids'][fi]
        if len(tids) < 2:
            continue
        for (i, b1), (j, b2) in combinations(enumerate(bboxes), 2):
            if i >= len(tids) or j >= len(tids) or tids[i] == tids[j]:
                continue
            iou = iou_xyxy(b1, b2)
            if iou > best_iou:
                best_iou, best_pair = iou, (tids[i], tids[j])
    if best_iou > 0.01:
        return best_pair
    best_score = -1.0
    for f in window_frames:
        fi = frame_to_idx[f]
        bboxes, tids = det['bboxes'][fi], det['track_ids'][fi]
        if len(tids) < 2:
            continue
        for (i, b1), (j, b2) in combinations(enumerate(bboxes), 2):
            if i >= len(tids) or j >= len(tids) or tids[i] == tids[j]:
                continue
            dist  = float(np.linalg.norm(center(b1) - center(b2)))
            score = 1.0 / (dist + 1.0)
            if score > best_score:
                best_score, best_pair = score, (tids[i], tids[j])
    return best_pair

def get_bbox_at_frame(track, target_frame):
    if track is None:
        return None
    for i, f in enumerate(track['frames']):
        if f >= target_frame:
            return track['bboxes'][i]
    return track['bboxes'][-1] if track['bboxes'] else None

def get_pre_collision_velocity(track, acc_frame, fps, lookback=1.5):
    lb = int(lookback * fps)
    vels = [track['velocities'][i]
            for i, f in enumerate(track['frames'][:-1])
            if (acc_frame - lb) <= f < acc_frame and i < len(track['velocities'])]
    if not vels:
        return track['velocities'][-1] if track['velocities'] else (0, 0)
    return tuple(np.mean(vels, axis=0))

def classify_from_trajectories(det, trajectories, accident_time, fps):
    acc_frame      = int(accident_time * fps)
    colliding_pair = find_colliding_pair(det, trajectories, acc_frame, fps)
    if colliding_pair is None:
        return "rear-end"
    tid1, tid2 = colliding_pair
    track1, track2 = trajectories.get(tid1), trajectories.get(tid2)
    if track1 is None or track2 is None:
        return "single"
    vel1 = get_pre_collision_velocity(track1, acc_frame, fps)
    vel2 = get_pre_collision_velocity(track2, acc_frame, fps)
    sp1, sp2 = np.hypot(*vel1), np.hypot(*vel2)
    if sp1 < SPEED_THRESHOLD and sp2 < SPEED_THRESHOLD:
        return "single"
    if sp1 < SPEED_THRESHOLD or sp2 < SPEED_THRESHOLD:
        mv  = vel1 if sp1 > sp2 else vel2
        ang = math.atan2(mv[1], mv[0])
        return "t-bone" if abs(math.sin(ang)) > 0.7 else "rear-end"
    dot       = vel1[0]*vel2[0] + vel1[1]*vel2[1]
    cos_angle = np.clip(dot / (sp1 * sp2 + 1e-8), -1, 1)
    rel_angle = math.degrees(math.acos(cos_angle))
    if rel_angle > 135:              return "head-on"
    elif rel_angle < 30:             return "rear-end"
    elif 60 <= rel_angle <= 120:     return "t-bone"
    elif 30 <= rel_angle < 60:       return "sideswipe"
    else:                            return "head-on"


# ─────────────────────────────────────────────────────────────
# CELL 6 — Run the full inference pipeline on your video
# ─────────────────────────────────────────────────────────────

print(f"📂 Loading video: {VIDEO_PATH}")
frames, fps = load_frames(VIDEO_PATH, max_seconds=MAX_VIDEO_SEC)
H, W = frames[0].shape[:2]
print(f"   {len(frames)} frames @ {fps:.1f} FPS  ({W}×{H})")

# Step 1 — Detection + Tracking
det, trajectories = run_tracking(frames, fps)

# Step 2 — Interaction scores
int_scores = compute_interaction_scores(det, trajectories, fps)
candidates = sorted(int_scores, key=lambda x: x[1], reverse=True)[:3]

# Step 3 — VideoMAE temporal localisation
cand_frames = [c[0] for c in candidates] if candidates else None
feats, stamps = build_feature_sequence(
    frames, fps, det, trajectories,
    clip_stride_sec=1.0, candidate_frames=cand_frames)
accident_time = tridet_predict_time(feats, stamps)
accident_time = float(np.clip(accident_time, 0, len(frames) / fps))

# Step 4 — Spatial localisation (impact point)
acc_frame      = int(np.clip(accident_time * fps, 0, len(frames) - 1))
colliding_pair = find_colliding_pair(det, trajectories, acc_frame, fps)
ip             = np.array([W * 0.5, H * 0.5], dtype=np.float32)

if colliding_pair is not None:
    tid1, tid2 = colliding_pair
    bb1 = get_bbox_at_frame(trajectories.get(tid1), acc_frame)
    bb2 = get_bbox_at_frame(trajectories.get(tid2), acc_frame)
    if bb1 is not None and bb2 is not None:
        ip = impact_point_from_boxes(bb1, bb2)

cx = float(np.clip(ip[0] / W, 0, 1))
cy = float(np.clip(ip[1] / H, 0, 1))

# Step 5 — Type classification
acc_type = classify_from_trajectories(det, trajectories, accident_time, fps)

print("\n" + "="*50)
print(f"  🕐 Accident time : {accident_time:.2f} s")
print(f"  📍 Coordinates   : ({cx:.4f}, {cy:.4f})  [normalised]")
print(f"  🚗 Type          : {acc_type}")
print("="*50)


# ─────────────────────────────────────────────────────────────
# CELL 7 — Render annotated output video
# ─────────────────────────────────────────────────────────────

OUTPUT_PATH = "accident_output.mp4"

# Pre-build a dict for fast bbox lookup per frame
frame_to_det_idx = {f: i for i, f in enumerate(det['frames'])}

# Colour palette for tracked vehicles
TRACK_COLOURS = {}
RNG = np.random.default_rng(42)
def track_colour(tid):
    if tid not in TRACK_COLOURS:
        TRACK_COLOURS[tid] = tuple(int(x) for x in RNG.integers(80, 255, 3))
    return TRACK_COLOURS[tid]

# Annotation parameters
ACCENT   = (0, 220, 255)          # yellow-ish (BGR)
RED      = (0, 60, 255)
WHITE    = (255, 255, 255)
FONT     = cv2.FONT_HERSHEY_DUPLEX
FONT_SM  = cv2.FONT_HERSHEY_SIMPLEX
RADIUS_IMPACT = max(18, W // 50)

# Pre-compute accident frame indicator window (±0.5s)
ACC_WINDOW = int(0.5 * fps)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (W, H))

print(f"Rendering {len(frames)} frames …")

for frame_idx, rgb in enumerate(tqdm(frames, desc="Writing output video")):
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

    # ── Draw bounding boxes for detected vehicles ────────────
    if frame_idx in frame_to_det_idx:
        di   = frame_to_det_idx[frame_idx]
        bbs  = det['bboxes'][di]
        tids = det['track_ids'][di]
        for bb, tid in zip(bbs, tids):
            x1, y1, x2, y2 = (int(v) for v in bb)
            col = track_colour(tid)
            cv2.rectangle(bgr, (x1, y1), (x2, y2), col, 2)
            label = f"ID {tid}"
            (tw, th), _ = cv2.getTextSize(label, FONT_SM, 0.55, 1)
            cv2.rectangle(bgr, (x1, y1 - th - 4), (x1 + tw + 4, y1), col, -1)
            cv2.putText(bgr, label, (x1 + 2, y1 - 2), FONT_SM, 0.55, (0, 0, 0), 1, cv2.LINE_AA)

    # ── Accident zone highlight ───────────────────────────────
    near_accident = abs(frame_idx - acc_frame) <= ACC_WINDOW

    if near_accident:
        # Red vignette overlay
        overlay = bgr.copy()
        cv2.rectangle(overlay, (0, 0), (W, H), (0, 0, 180), -1)
        alpha = 0.18 * max(0, 1 - abs(frame_idx - acc_frame) / ACC_WINDOW)
        bgr = cv2.addWeighted(bgr, 1 - alpha, overlay, alpha, 0)

        # Impact point crosshair
        px, py = int(ip[0]), int(ip[1])
        cv2.circle(bgr, (px, py), RADIUS_IMPACT, RED, 3)
        cv2.line(bgr, (px - RADIUS_IMPACT, py), (px + RADIUS_IMPACT, py), RED, 2)
        cv2.line(bgr, (px, py - RADIUS_IMPACT), (px, py + RADIUS_IMPACT), RED, 2)
        cv2.putText(bgr, "IMPACT", (px + RADIUS_IMPACT + 4, py - 4),
                    FONT_SM, 0.6, RED, 2, cv2.LINE_AA)

    # ── HUD — top-left panel ──────────────────────────────────
    panel_h, panel_w = 110, 380
    panel = np.zeros((panel_h, panel_w, 3), dtype=np.uint8)
    panel[:] = (20, 20, 20)

    cur_t = frame_idx / fps
    cv2.putText(panel, f"Time:      {cur_t:6.2f} s", (10, 22),  FONT_SM, 0.58, WHITE, 1, cv2.LINE_AA)
    cv2.putText(panel, f"Acc. time: {accident_time:.2f} s",      (10, 46),  FONT_SM, 0.58, ACCENT, 1, cv2.LINE_AA)
    cv2.putText(panel, f"Coords:    ({cx:.4f}, {cy:.4f})",       (10, 70),  FONT_SM, 0.58, ACCENT, 1, cv2.LINE_AA)
    cv2.putText(panel, f"Type:      {acc_type.upper()}",          (10, 94),  FONT_SM, 0.62, ACCENT, 2, cv2.LINE_AA)

    bgr[10:10 + panel_h, 10:10 + panel_w] = (
        cv2.addWeighted(bgr[10:10 + panel_h, 10:10 + panel_w], 0.25, panel, 0.75, 0)
    )

    # ── "ACCIDENT DETECTED" banner during accident window ─────
    if near_accident:
        banner = "⚠  ACCIDENT DETECTED"
        (bw, bh), _ = cv2.getTextSize(banner, FONT, 0.9, 2)
        bx = (W - bw) // 2
        by = H - 40
        cv2.rectangle(bgr, (bx - 8, by - bh - 6), (bx + bw + 8, by + 6), (0, 0, 200), -1)
        cv2.putText(bgr, banner, (bx, by), FONT, 0.9, WHITE, 2, cv2.LINE_AA)

    writer.write(bgr)

writer.release()
print(f"\n✅ Output video saved: {OUTPUT_PATH}")


# ─────────────────────────────────────────────────────────────
# CELL 8 — Download the output video
# ─────────────────────────────────────────────────────────────
from google.colab import files
files.download(OUTPUT_PATH)
print("📥 Download started — check your browser's downloads folder.")

Loading GroundingDINO …


Loading weights:   0%|          | 0/1206 [00:00<?, ?it/s]

Loading VideoMAE-v2 …


Loading weights:   0%|          | 0/184 [00:00<?, ?it/s]

VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     |  | 
---------------------------------------------------------------------+------------+--+-
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.q_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.query.weight | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_after.weight           | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.bias            | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.attention.v_bias       | UNEXPECTED |  | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED |  | 
decoder.norm.bias                                                 

...

Writing output video: 100%|██████████| 431/431 [00:03<00:00, 133.62it/s]


✅ Output video saved: accident_output.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started — check your browser's downloads folder.
